In [1]:
import pandas as pd
import numpy as np
from pulp import *
from pathlib import Path

In [2]:
DATA_DIR = Path('../data/network_analysis')

network_df_path= DATA_DIR / 'network_df.parquet'
grid_constraint_df_path= DATA_DIR / 'example_grid_constraint_df.parquet'
feeder_block_matrix_path= DATA_DIR / 'feeder_block_matrix.parquet'

In [3]:
network_df = pd.read_parquet(network_df_path)
grid_df = pd.read_parquet(grid_constraint_df_path)
feeder_block_df = pd.read_parquet(feeder_block_matrix_path)

In [ ]:
origin_nodes = network_df['geoid'].unique()
dest_nodes = network_df['neighbor_geoid'].unique()
nodes = list(set(origin_nodes) | set(dest_nodes))
print(f"Total nodes (census block groups): {len(nodes)}")

# Create node demand dictionary
node_demand = {}
for node in nodes:
    # Get demand from origin_demand column
    demand_rows = network_df[network_df['geoid'] == node]
    if len(demand_rows) > 0:
        node_demand[node] = demand_rows['origin_demand_(kW)'].iloc[0]
    else:
        node_demand[node] = 0.0

# Create edges list with costs
edges = []
for idx, row in network_df.iterrows():
    origin = row['geoid']
    neighbor = row['neighbor_geoid']
    # Cost is based on distance (could be weighted differently)
    cost = row['distance_km']
    edges.append((origin, neighbor, cost))

print(f"Total edges: {len(edges)}")

# Prepare feeder data
feeders = list(grid_df['feeder_id'].values)
feeder_capacity = dict(zip(
    grid_df['feeder_id'], 
    grid_df['available_capacity']
))

print(f"Total feeders: {len(feeders)}")

# Create feeder-node mapping from matrix
# The matrix has feeders as rows and GEOIDs as columns
feeder_node_proportion = {}
nodes_represented = set()
for feeder_id in feeder_block_df.index:
    for geoid in feeder_block_df.columns:
        proportion = feeder_block_df.loc[feeder_id, geoid]
        if proportion > 0:
            if feeder_id not in feeder_node_proportion:
                feeder_node_proportion[feeder_id] = {}
            feeder_node_proportion[feeder_id][geoid] = proportion
            nodes_represented.add(geoid)

print(f"Feeder-node mappings created: {len(feeder_node_proportion)}")
print(f"Nodes not mapped to feeders: {len(set(nodes) - nodes_represented)} of {len(nodes)}")


Total nodes (census block groups): 1134
Total edges: 7276
Total feeders: 1969
Feeder-node mappings created: 3023
